## Transformers END-To-END

## English ---> French

In [82]:
data = [
    ("i am a student", "je suis un etudiant"),
    ("how are you", "comment allez vous"),
    ("i love machine learning", "j aime apprentissage automatique"),
    ("good morning", "bonjour"),
    ("thank you", "merci"),
    ("see you later", "a plus tard"),
    ("what is your name", "quel est votre nom"),
    ("where are you going", "ou allez vous"),
    ("i like coffee", "j aime le cafe"),
    ("welcome", "bienvenue")
]

In [83]:
# Import necessay libraries
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import(
    TextVectorization, 
    Embedding, 
    Dense, 
    LayerNormalization, 
    MultiHeadAttention
)

from tensorflow.keras import Model

In [84]:
# Separate Input and Output Sentences

english_sentences = [x[0] for x in data]

# add START and END tokens

french_sentences = [
    "start " + x[1] + " end"
    for x in data
]

In [85]:
# Tokenization
vocab_size = 1000 # keep a maximum of 1000 unique words in vocabulary
sequence_length = 20 # Maximum length of each sentence 

In [86]:
# Tokenize English Sentences
source_vectorization = TextVectorization(
    max_tokens = vocab_size,
    output_mode = "int",
    output_sequence_length = sequence_length
)

In [87]:
# Tokenize English Sentences
target_vectorization = TextVectorization(
    max_tokens = vocab_size,
    output_mode = "int",
    output_sequence_length = sequence_length
)

In [88]:
# This is where learning happens. (Adapt the tokenizers to the data)

source_vectorization.adapt(
    english_sentences
)

target_vectorization.adapt(
    french_sentences
)

In [89]:
# convert Text into Numbers 
encoder_inputs = source_vectorization(english_sentences)
target_tokens = target_vectorization(french_sentences)

In [90]:
# Prepare Decoder Input & Output
decoder_inputs = target_tokens[:, :-1]
decoder_targets = target_tokens[:, 1:]

In [91]:
# Positional Encoding

class PositionalEmbedding(tf.keras.layers.Layer):
    # Constructor
    def __init__(self, 
                 sequence_length, 
                 vocab_size, 
                 embed_dim):
        # Call the parent constructor
        super().__init__()
        # Token Embedding
        self.token_embedding = Embedding(
            vocab_size,
            embed_dim
        )
        self.position_embedding = Embedding(
            sequence_length,
            embed_dim
        )
        # Store sequence length for later use
        self.sequence_length = sequence_length
    
    # Call method to compute the input sequence
    def call(self, inputs):
        # Get the length of the input sequence
        length = tf.shape(inputs)[-1]
        # create position numbers
        positions = tf.range(
            start=0,
            limit=length,
            delta=1
        )

        # convert words to embeddings
        embedded_tokens = self.token_embedding(inputs)

        # Convert positions to embeddings
        embedded_positions = self.position_embedding(positions)

        # Add both embedding togather
        return embedded_tokens + embedded_positions

In [92]:
# Encoder Block

# Create a custom Encoder Layer
class TransformerEncoder(tf.keras.layers.Layer):
    # Constructor
    # embed_dim : 128, dense_dim : 512, num_heads : 4
    # Head1 --> Grammar, Head2 --> Context, Head3 --> Relationships, Heard4 --> Meaning

    def __init__(self, embed_dim, 
                 dense_dim, 
                 num_heads) :
        super().__init__()
        self.attention = MultiHeadAttention(
            num_heads = num_heads,
            key_dim = embed_dim
        )
        # Feed Forward Network
        # Attentation mixes information.
        # FFN learns complex features.
        # I love ai (Input) --> Attention --> AI is related to Love --> FFN --> (AI is the object being loved) i love AI (output)

        self.dense_proj = tf.keras.Sequential([
            Dense(
                dense_dim,
                activation='relu'
            ),
            Dense(embed_dim)
        ])  
        # Layer Normalization

        self.layernorm1 = LayerNormalization()
        self.layernorm2 = LayerNormalization()

    def call(self, inputs):
        # self attention
        attention_output = self.attention(
            inputs,
            inputs
        )
        # Residual Connection + LayerNorm
        proj_input = self.layernorm1(
            inputs + attention_output
        )

        # Feed forward neural network
        proj_output = self.dense_proj(
            proj_input
        )

        # Second Residual Connection + LayerNorm

        return self.layernorm2(
              proj_input + proj_output
        )

In [93]:
# Decoder Block

class TransformerDecoder(tf.keras.layers.Layer):
    
    def __init__(self, 
                 embed_dim, 
                 dense_dim, 
                 num_heads) :
        super().__init__()

        self.self_attention = MultiHeadAttention(
            num_heads = num_heads,
            key_dim = embed_dim
        )

        self.cross_attention = MultiHeadAttention(
            num_heads = num_heads,
            key_dim = embed_dim
        )

        self.ffn = tf.keras.Sequential(
            [
                Dense(
                    dense_dim,
                    activation = "relu"
                ),
                Dense(embed_dim)
            ]
        )

        # Layer Normalization
        self.layernorm1 = LayerNormalization()
        self.layernorm2 = LayerNormalization()
        self.layernorm3 = LayerNormalization()
    
    def call(
            self,
            inputs,
            encoder_outputs
    ):
        # masked self attention

        attention_output = self.self_attention(
            query = inputs,
            value = inputs,
            key = inputs,
            use_causal_mask = True
        )

        out1 = self.layernorm1(
            inputs + attention_output
        )

        # Cross Attention

        attention_output2 = self.cross_attention(
            out1 , 
            encoder_outputs,
        )

        out2 = self.layernorm1(
            out1 + attention_output2
        )

        # Feed Forward neural network
        ffn_output = self.ffn(out2)

        return self.layernorm3(
            out2 + ffn_output
        )


In [94]:
# Build Complete Transformer 
embed_dim = 128
dense_dim = 256
num_heads = 4

# Encoder input layer 
encoder_input = tf.keras.Input(
    shape = (None,),
    dtype = "int64"
)

x = PositionalEmbedding(
    sequence_length,
    vocab_size, 
    embed_dim
)(encoder_input)

# Encoder Block 
encoder_output = TransformerEncoder(
    embed_dim,
    dense_dim,
    num_heads
)(x)

# Decoder Block 
decoder_input = tf.keras.Input(
    shape = (None,),
    dtype = "int64"
)

x = PositionalEmbedding(
    sequence_length,
    vocab_size,
    embed_dim
)(decoder_input)

x = TransformerDecoder(
    embed_dim,
    dense_dim,
    num_heads
)(x, encoder_output)

decoder_output = Dense(
    vocab_size,
    activation = "softmax"
)(x)

# create model 
transformer = Model(
    [encoder_input, decoder_input],
    decoder_output
)

/Users/santhoshib/miniconda3/envs/cnn/lib/python3.11/site-packages/keras/src/layers/layer.py:427: UserWarning: `build()` was called on layer 'transformer_decoder_6', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


In [95]:
# create model 
transformer = Model(
    [encoder_input, decoder_input],
    decoder_output
)
transformer.compile(
    optimizer = "adam",
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)

transformer.fit(
    [encoder_inputs, decoder_inputs],
    decoder_targets,
    batch_size = 2,
    epochs = 30
)

Epoch 1/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.6368 - loss: 4.3659   
Epoch 2/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8000 - loss: 2.3257
Epoch 3/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8000 - loss: 1.7122
Epoch 4/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8000 - loss: 1.3610
Epoch 5/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8000 - loss: 1.1545
Epoch 6/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8158 - loss: 0.9406
Epoch 7/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.8421 - loss: 0.7715
Epoch 8/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.8368 - loss: 0.6422
Epoch 9/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8632 - loss: 0.5475
Epoch 10/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8947 - loss: 0.4693
Epoch 11/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8947 - loss: 0.3874
Epoch 12/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9421 - loss: 0.331

In [96]:
# Prediction
 
test_sentence = ["i like coffee"]
 
encoder_input_test = source_vectorization(
    test_sentence
)
 
start_sentence = "start"

decoded_sentence = start_sentence
 
for i in range(10):
 
    tokenized_target = target_vectorization(
    [decoded_sentence]
    )[:, :-1]
 
    predictions = transformer.predict(
        [
            encoder_input_test,
            tokenized_target
        ],
        verbose=0
    )
 
    sampled_token_index = np.argmax(
        predictions[0, i, :]
    )
 
    index_lookup = dict(
        zip(
            range(
                len(
                    target_vectorization.get_vocabulary()
                )
            ),
            target_vectorization.get_vocabulary()
        )
    )
 
    sampled_token = index_lookup[
        sampled_token_index
    ]
 
    decoded_sentence += " " + sampled_token
 
    if sampled_token == "end":
        break
 
print(decoded_sentence)


start j aime le cafe end
